# Step 1: people and sky head on SAM 2.1 tiny `image_embed` (Colab)

README section 5, step 1. Scripts: `scripts/poc/people_sky_head/` (see its README for the
objective and the measurements).

Runtime: **GPU** (L4 or better, high RAM). The run downloads the COCO 2017 panoptic annotations and
the Open Images V7 annotations, and only the photos with a license that permits commercial use
(README decision U2). It caches `image_embed` (2 MiB per photo) on the local disk of the runtime,
trains the head on COCO and Open Images, and writes the checkpoints and `people_sky_head.onnx` to
`DRIVE_OUT/RUN_NAME/` on Google Drive. If the runtime is reset, run all cells again: the downloads
and the encoder continue where they stopped while the runtime disk exists; training continues
with `RESUME = True`.

With `TRAIN = False`, the notebook prepares and encodes only the validation splits and measures
the checkpoints of `EVALUATE_CHECKPOINTS` with `evaluate_head.py`.

In [ ]:
# Settings.
REPO_URL = "https://github.com/zidage/alcedo-ai-mask-lab.git"
REPO_BRANCH = "main"
SAM2_COMMIT = "2b90b9f5ceec907a1c18123530e92e794ad901a4"
CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt"

# COCO license ids: 1 BY-NC-SA, 2 BY-NC, 3 BY-NC-ND, 4 BY, 5 BY-SA, 6 BY-ND,
# 7 no known copyright restrictions, 8 United States Government Work.
# Training uses only licenses that permit commercial use (README decision U2).
TRAIN_LICENSES = "4,5,7,8"
TRAIN_MAX_IMAGES = 0      # 0 = all images with an allowed license
VAL_LICENSES = "all"      # validation photos are only measured, not trained on
VAL_MAX_IMAGES = 2500

# Open Images V7: annotations CC BY 4.0; only photos listed as CC BY 2.0 and with rotation 0.
# Per split: photos with Sky or Cloud point labels, then photos with person boxes.
OI_LICENSES = "https://creativecommons.org/licenses/by/2.0/"
OI_TRAIN_SKY_IMAGES = 12000
OI_TRAIN_PERSON_IMAGES = 12000
OI_VAL_SKY_IMAGES = 1250
OI_VAL_PERSON_IMAGES = 1250

EPOCHS = 20
BATCH_SIZE = 32
RESUME = False
# False: no train splits and no training; only measure EVALUATE_CHECKPOINTS on the validation
# splits (evaluate_head.py) with the measurements of the current head_metrics.py.
TRAIN = True

WORK = "/content/work"
DRIVE_OUT = "/content/drive/MyDrive/alcedo_people_sky_head"
RUN_NAME = "run2_coco_openimages"          # this training run: DRIVE_OUT/RUN_NAME/{run,export,splits}
RUN_DIR = f"{DRIVE_OUT}/{RUN_NAME}"
# Checkpoints that are measured in addition to the best.pt of this run (the first run, COCO only).
EVALUATE_CHECKPOINTS = [f"{DRIVE_OUT}/run/best.pt"]

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content
!free -g

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_OUT, exist_ok=True)

In [ ]:
import os
os.makedirs(WORK, exist_ok=True)
%cd {WORK}
if not os.path.isdir("alcedo-ai-mask-lab"):
    !git clone --branch {REPO_BRANCH} {REPO_URL} alcedo-ai-mask-lab
else:
    !git -C alcedo-ai-mask-lab fetch origin {REPO_BRANCH} && git -C alcedo-ai-mask-lab checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}
if not os.path.isdir("sam2"):
    !git clone https://github.com/facebookresearch/sam2.git sam2
!git -C sam2 checkout -q {SAM2_COMMIT} && git -C sam2 rev-parse HEAD
!git -C alcedo-ai-mask-lab log -1 --oneline
!pip install -q -r alcedo-ai-mask-lab/scripts/poc/people_sky_head/requirements_colab.txt
!wget -q -nc {CHECKPOINT_URL}; sha256sum sam2.1_hiera_tiny.pt

## COCO 2017 panoptic annotations

`panoptic_annotations_trainval2017.zip` (person and `sky-other-merged` segments) and the caption
files of `annotations_trainval2017.zip` (the `images` list with the Flickr license of each photo).
COCO annotations are CC BY 4.0.

In [ ]:
%cd {WORK}
!mkdir -p coco/annotations
!wget -q -nc http://images.cocodataset.org/annotations/panoptic_annotations_trainval2017.zip
!wget -q -nc http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q -n -j panoptic_annotations_trainval2017.zip -d coco/annotations
!unzip -q -n -j annotations_trainval2017.zip annotations/captions_train2017.json annotations/captions_val2017.json -d coco/annotations
!cd coco && unzip -q -n annotations/panoptic_train2017.zip && unzip -q -n annotations/panoptic_val2017.zip
!ls coco coco/annotations && ls coco/panoptic_train2017 | wc -l

## Open Images V7 annotations

Point labels (Sky and Cloud), box annotations (Person, Man, Woman, Boy, Girl), human-verified
image-level labels, and the image list with the license and the rotation of each photo.
Annotations CC BY 4.0 (Google LLC). The training files are about 11 GB; the validation files
are about 160 MB.

In [ ]:
%cd {WORK}
!mkdir -p openimages
OI = "https://storage.googleapis.com/openimages"
OI_FILES = {
    "validation": {"points": f"{OI}/v7/oidv7-val-annotations-point-labels.csv",
                   "boxes": f"{OI}/v5/validation-annotations-bbox.csv",
                   "labels": f"{OI}/v7/oidv7-val-annotations-human-imagelabels.csv",
                   "images": f"{OI}/2018_04/validation/validation-images-with-rotation.csv"},
    "train": {"points": f"{OI}/v7/oidv7-train-annotations-point-labels.csv",
              "boxes": f"{OI}/v6/oidv6-train-annotations-bbox.csv",
              "labels": f"{OI}/v7/oidv7-train-annotations-human-imagelabels.csv",
              "images": f"{OI}/v6/oidv6-train-images-with-labels-with-rotation.csv"},
}
for oi_split in ("validation", "train") if TRAIN else ("validation",):
    for url in OI_FILES[oi_split].values():
        !wget -q -nc -P openimages {url}
!ls -la openimages

## Targets and photos

Selects the photos by license, downloads only these photos, and writes the per-cell targets.
The first lines of each output list the photos per license (COCO) or the selection pools and the
rejected candidates (Open Images).

In [ ]:
%cd {WORK}
S = "alcedo-ai-mask-lab/scripts/poc/people_sky_head"
SPLITS = (("train", TRAIN_LICENSES, TRAIN_MAX_IMAGES),) if TRAIN else ()
SPLITS += (("val", VAL_LICENSES, VAL_MAX_IMAGES),)
for split, licenses, max_images in SPLITS:
    !python {S}/prepare_coco_targets.py         --panoptic-json coco/annotations/panoptic_{split}2017.json         --panoptic-png-dir coco/panoptic_{split}2017         --image-info-json coco/annotations/captions_{split}2017.json         --image-dir coco/{split}2017 --licenses {licenses} --max-images {max_images}         --out-dir splits/{split} --workers 2
ENCODE_SPLITS = [(split, f"coco/{split}2017") for split, _, _ in SPLITS]

In [ ]:
%cd {WORK}
OI_SPLITS = (("oi_train", "train", OI_TRAIN_SKY_IMAGES, OI_TRAIN_PERSON_IMAGES),) if TRAIN else ()
OI_SPLITS += (("oi_val", "validation", OI_VAL_SKY_IMAGES, OI_VAL_PERSON_IMAGES),)
for name, oi_split, sky_images, person_images in OI_SPLITS:
    images_csv, points_csv, boxes_csv, labels_csv = (
        "openimages/" + OI_FILES[oi_split][k].rsplit("/", 1)[1]
        for k in ("images", "points", "boxes", "labels"))
    !python {S}/prepare_openimages_targets.py --images-csv {images_csv} --points-csv {points_csv} --boxes-csv {boxes_csv} --image-labels-csv {labels_csv} --split {oi_split} --image-dir openimages/{oi_split} --licenses {OI_LICENSES} --sky-images {sky_images} --person-images {person_images} --out-dir splits/{name}
    if (name, f"openimages/{oi_split}") not in ENCODE_SPLITS:
        ENCODE_SPLITS.append((name, f"openimages/{oi_split}"))
print(ENCODE_SPLITS)

## image_embed cache

SAM 2.1 tiny encoder (float32, TF32 off), float16 cache: 2 MiB per photo. The next cell checks
that the local disk has space for both caches before the encoder starts (all splits).

In [ ]:
import json, shutil
need = 0
for split, _ in ENCODE_SPLITS:
    count = len(json.load(open(f"{WORK}/splits/{split}/manifest.json"))["images"])
    need += count * 256 * 64 * 64 * 2
free = shutil.disk_usage(WORK).free
print(f"cache {need / 2**30:.1f} GiB, free {free / 2**30:.1f} GiB")
if need > free - 2 * 2**30:
    raise RuntimeError("not enough local disk for the image_embed cache: lower the OI_* image "
                       "counts, TRAIN_MAX_IMAGES, or VAL_MAX_IMAGES, or use a runtime "
                       "with a larger disk")

In [ ]:
%cd {WORK}
for split, image_dir in ENCODE_SPLITS:
    !python {S}/encode_image_embed.py --split-dir splits/{split} --image-dir {image_dir} --sam2-repo sam2 --checkpoint sam2.1_hiera_tiny.pt --batch-size 8 --workers 2

## Training

The training set is COCO `train` and Open Images `oi_train`. `best.pt` is selected on COCO `val`;
`oi_val` is measured each epoch (`val_extra` in `metrics.jsonl`). Checkpoints and `metrics.jsonl`
go to `RUN_DIR/run` on Google Drive. Set `RESUME = True` in the settings cell to continue from
`last.pt` after a reset.

In [ ]:
%cd {WORK}
resume = "--resume" if RESUME else ""
TRAIN_DIRS = " ".join(f"splits/{s}" for s, _ in ENCODE_SPLITS if s in ("train", "oi_train"))
VAL_DIRS = " ".join(f"splits/{s}" for s, _ in ENCODE_SPLITS if s in ("val", "oi_val"))
if TRAIN:
    !python {S}/train_head.py --train-dir {TRAIN_DIRS} --val-dir {VAL_DIRS} --out-dir {RUN_DIR}/run --epochs {EPOCHS} --batch-size {BATCH_SIZE} --workers 2 {resume}

In [ ]:
import json, os
def show(value):
    return "-" if value is None else f"{value:.3f}"
if os.path.exists(f"{RUN_DIR}/run/metrics.jsonl"):
    for line in open(f"{RUN_DIR}/run/metrics.jsonl"):
        m = json.loads(line); v = m["val"]
        print(f"epoch {m['epoch']:2d} loss {m['train_loss']:.4f} | COCO sky IoU {v['sky_cell_iou']:.3f} "
              f"AP50 {v['person_ap50']:.3f} top-1 {v['person_top1_hit']:.3f} presence AUC sky "
              f"{v['presence_auc_sky']:.3f} person {v['presence_auc_person']:.3f}")
        for name, x in m.get("val_extra", {}).items():
            print(f"         {name}: sky point AUC {show(x['sky_point_auc'])} false sky "
                  f"{show(x['sky_point_false_sky_share'])} AP50 {show(x['person_ap50'])} "
                  f"presence AUC person {show(x['presence_auc_person'])}")

## Export

`people_sky_head.onnx` and `people_sky_head.json` (hash, validation measurements, the ONNX
Runtime against PyTorch comparison) go to `RUN_DIR/export`. The split manifests (photo ids,
licenses, authors) are copied to `RUN_DIR/splits` as the record of the training data.

In [ ]:
%cd {WORK}
if TRAIN:
    !python {S}/export_head_onnx.py --checkpoint {RUN_DIR}/run/best.pt --val-dir splits/val --out-dir {RUN_DIR}/export
    for split, _ in ENCODE_SPLITS:
        !mkdir -p {RUN_DIR}/splits/{split} && cp splits/{split}/manifest.json splits/{split}/image_embed_progress.json {RUN_DIR}/splits/{split}/
    !ls -la {RUN_DIR}/export {RUN_DIR}/run

## Measurements of the checkpoints

`evaluate_head.py` measures `RUN_DIR/run/best.pt` (in a training run) and the checkpoints of
`EVALUATE_CHECKPOINTS` on COCO `val` and Open Images `oi_val` with the current `head_metrics.py`:
the People list threshold after the suppression of overlapping boxes (`person_list_best_f1_nms`),
the box fit per person height (`person_box_fit_nms`), and the sky point measurements
(`sky_point_*`, Open Images only). Each result goes to `<checkpoint>_val_measurements.json` next
to the checkpoint.

In [ ]:
%cd {WORK}
VAL_DIRS = " ".join(f"splits/{s}" for s, _ in ENCODE_SPLITS if s in ("val", "oi_val"))
for checkpoint in ([f"{RUN_DIR}/run/best.pt"] if TRAIN else []) + EVALUATE_CHECKPOINTS:
    out = checkpoint[:-len(".pt")] + "_val_measurements.json"
    !python {S}/evaluate_head.py --checkpoint {checkpoint} --val-dir {VAL_DIRS} --out {out}